<a href="https://colab.research.google.com/github/oj-twri/fpl-predictor/blob/main/02_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# Import the libraries we need
import pandas as pd
import numpy as np

# Load one past season of gameweek data from the vaastav repo
# (skip this cell if `hist` is already loaded in your current notebook)
url = "https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/master/data/2023-24/gws/merged_gw.csv"
hist = pd.read_csv(url, encoding="latin-1")

# Check the size: expect roughly 25,000 rows
print(hist.shape)

(29725, 41)


In [10]:
# Sort by player, then gameweek, so each player's games are in chronological order
# (rolling averages only make sense if the rows are in the right order)
hist = hist.sort_values(["element", "GW", "kickoff_time"]).reset_index(drop=True)

# Quick look to confirm the sorting worked
# 'element' is the player's ID
print(hist.shape)
hist[["element", "name", "GW", "total_points"]].head()

(29725, 41)


,element,name,GW,total_points
0,1,Folarin Balogun,1,0
1,1,Folarin Balogun,2,0
2,1,Folarin Balogun,3,0
3,1,Folarin Balogun,4,0
4,1,Folarin Balogun,5,0


In [11]:
# Create two baseline predictions: average of the last 3 games and the last 5 games
for window in [3, 5]:
    hist[f"pred_last{window}"] = (
        hist.groupby("element")["total_points"]   # handle each player separately
        .transform(
            # shift(1) moves every value down one row, so a game's prediction
            # only uses games BEFORE it (never the game being predicted)
            # rolling(window) then averages the previous 'window' games
            # min_periods=1 lets it use fewer games early in the season
            lambda s: s.shift(1).rolling(window, min_periods=1).mean()
        )
    )

# Check one player: GW1 should be NaN (no history),
# and GW2's prediction should equal his GW1 score
hist[hist["name"].str.contains("Isak")][["name", "GW", "total_points", "pred_last3", "pred_last5"]].head(10)

,name,GW,total_points,pred_last3,pred_last5
15733,Alexander Isak,1,13,NaN,NaN
15734,Alexander Isak,2,2,13.000000,13.000000
15735,Alexander Isak,3,2,7.500000,7.500000
15736,Alexander Isak,4,2,5.666667,5.666667
15737,Alexander Isak,5,0,2.000000,4.750000
15738,Alexander Isak,6,5,1.333333,3.800000
15739,Alexander Isak,7,6,2.333333,2.200000
15740,Alexander Isak,8,13,3.666667,3.000000
15741,Alexander Isak,9,1,8.000000,5.200000
15742,Alexander Isak,10,0,6.666667,5.000000


In [12]:
# Drop rows with no prediction (a player's very first game has no history)
eval_df = hist.dropna(subset=["pred_last3", "pred_last5"])

# Helper function: given a prediction column, return its error scores
def report(df, col):
    err = df["total_points"] - df[col]      # actual minus predicted
    return {
        "MAE": err.abs().mean(),             # average miss, in points
        "RMSE": np.sqrt((err ** 2).mean())   # like MAE but punishes big misses more
    }

# Score both baselines and show them side by side
results = pd.DataFrame({c: report(eval_df, c) for c in ["pred_last3", "pred_last5"]}).T
results

,MAE,RMSE
pred_last3,1.009667,2.144573
pred_last5,0.994255,2.075514


In [13]:
# Compare against the simplest possible prediction: everyone scores the average
mean_pts = eval_df["total_points"].mean()

# MAE if we guessed the same number for every player
# (slightly flattering to the dumb guess, since it uses the full-season average,
# but it gives you a floor to compare your baselines against)
print("Always predict the average:", (eval_df["total_points"] - mean_pts).abs().mean())

Always predict the average: 1.3959548882002104


In [14]:
# For picking a team, getting the ORDER of players right matters more than exact scores
# For each gameweek, compare how the prediction ranks players with how they actually scored
# Spearman correlation: 1 = perfect ordering, 0 = no relationship, -1 = reversed
rank_corr = (
    eval_df.groupby("GW")[["pred_last5", "total_points"]]
    .apply(lambda g: g.corr(method="spearman").iloc[0, 1])
)

# Average the per-gameweek correlations into one number
print("Average rank correlation:", rank_corr.mean())

Average rank correlation: 0.685345450330796


In [15]:
print("Always predict 0:", eval_df["total_points"].abs().mean())

Always predict 0: 1.0696812196812198
